# Проект по курсу "Рекомендательные системы"
  
Правила заполнения ноутбуков на авто-проверку:
- повторить окружение преподавателя
 Python 3.13.0

  ```bash
  pip install implicit==0.7.2 "rectools[all]==0.17.0" pandas==2.3.3 numpy==2.4.1 scipy==1.17.0  requests==2.32.5 catboost==1.2.8 scikit-learn==1.7.2 torch==2.10.0 torchvision==0.25.0
  '''
- все решение должно полностью помещаться в функцию solution(смотри пример). Если вы хотите реализовать дополнительные функции - поместите их в область видимости soluition. Нельзя использовать дополнительные файлы.
- не добавлять новые импорты и не использовать дополнительные библиотеки. В противном случае ноутбук не пройдёт проверку и получит `0` баллов
- не добавлять аргументов в solution
- писать код только между # CODE BEGIN и # CODE END
- не менять код преподавателя
- не добавлять новые ячейки
- следить, чтобы не было warning - они автоматом фейлят задание
- перед сдачей проверить, что весь ноутбук прогоняется от начала до конца и все тесты проходят
- data_path должен браться из переменной окружения как в коде ниже
- Код должен выполняться за разумное время - ограничение 20 мин на 4 CPU и 16 Gb RAM без GPU. Не нужно ставить огромное количество эпох.
- Постарайтесь максимально зафиксировать сиды, чтобы не было сюрпризов во время автоматической проверки. В случае, если решение выдает разное качество при разных запусках, то в зачет идет то значение, которое получилось при автоматической проверке.


В данном проекте вам возможно захочется подбирать гипер-параметры моделей. Писать код для подбора гипер-параметров, использовать optuna и т.п. рекомендуем в отдельном ноутбуке.

Библиотеки implicit и lightfm не фиксируют random state при num_threads > 1. Если результат работы модели не сильно превышает  необходимый порог и рандом может опустить его ниже требуемого уровня, рекомендуем продолжить повышение качества модели: тюнинг гипер-параметров, подбор фичей, подбор метода обработки датасета

# Задание

Вам предлагается реализовать рекомендательную систему для фильмов KION.
Решение должно быть полностью упаковано в функцию solution. Качество будет проверяться с помощью метрики MAP@10 на отложенной неделе. Итоговый бал определеятся функцией scorer - вы можете посмотреть его сразу, но если модель не подразумевает фиксирование random state, то после прогона автоматической системой результат может немного отличаться.

В качестве примера реализована базовая рекомендательная система на основе ease. Ваша задача - улучшить эту систему.

Чтобы решение отрабатывало быстрее будем использовать 10% от общего числа пользователей.

В случае, если в вашем решении будет найден Hardcode элементов тестового датафрейма - работа будет аннулирована.

Успехов!

Подсказки:
- Можно посмотреть документацию rectools
- Можно поссмотреть ноутбуки с семинаров и предыдущую версию проекта
- Не стесняйтесь добавлять фичи в ранжирование
- Скорее всего вам понадобятся как отбор кандидатов, так и ранжирование

## Импорты и данные

In [ ]:
!python -V

In [ ]:
# Make sure you don’t add any new imports to the notebook. The solution must be limited to the provided libraries.

import os
os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = "1"

import warnings
warnings.simplefilter("ignore")

import implicit
import rectools
import pandas as pd
import numpy as np
import scipy
import requests
import catboost
import sklearn
import torch
from torch import nn

from rectools import models
from rectools import dataset
from rectools import metrics

print(implicit.__version__)
print(rectools.__version__)
print(pd.__version__)
print(np.__version__)
print(scipy.__version__)
print(requests.__version__)
print(catboost.__version__)
print(sklearn.__version__)
print(torch.__version__)

In [ ]:
import os.path

# For implicit ALS
import threadpoolctl
os.environ["OPENBLAS_NUM_THREADS"] = "1"
threadpoolctl.threadpool_limits(1, "blas")

Если у вас нет данных, то используйте закомментированный код

In [ ]:
# from tqdm.auto import tqdm
# import zipfile as zf

# url = 'https://github.com/irsafilo/KION_DATASET/raw/f69775be31fa5779907cf0a92ddedb70037fb5ae/data_original.zip'

# req = requests.get(url, stream=True)

# with open('kion.zip', 'wb') as fd:
#     total_size_in_bytes = int(req.headers.get('Content-Length', 0))
#     progress_bar = tqdm(desc='kion dataset download', total=total_size_in_bytes, unit='iB', unit_scale=True)
#     for chunk in req.iter_content(chunk_size=2 ** 20):
#         progress_bar.update(len(chunk))
#         fd.write(chunk)

# files = zf.ZipFile('kion.zip', 'r')
# files.extractall()
# files.close()

In [ ]:
data_path = os.environ.get("DATA_PATH")
if data_path is None:
    data_path = "data_original"  # ваш путь к данным до папки data_original включительно (поменяйте при необходимости)

In [ ]:
users = pd.read_csv(os.path.join(data_path, "users.csv"))
items = pd.read_csv(os.path.join(data_path, "items.csv"))

users = users.sample(frac=0.1, random_state=42)

interactions = (
    pd.read_csv(os.path.join(data_path, "interactions.csv"), parse_dates=["last_watch_dt"])
    .rename(columns={'total_dur': rectools.Columns.Weight,
                     'last_watch_dt': rectools.Columns.Datetime})
)


interactions = interactions[interactions["user_id"].isin(users["user_id"])]


print(interactions.shape)
interactions.head(5)

In [ ]:
N_DAYS = 7

max_date = interactions['datetime'].max()
train = interactions[(interactions['datetime'] <= max_date - pd.Timedelta(days=N_DAYS))]
test = interactions[(interactions['datetime'] > max_date - pd.Timedelta(days=N_DAYS))]

catalog = train[rectools.Columns.Item].unique()

test_users = test[rectools.Columns.User].unique()
cold_users = set(test_users) - set(train[rectools.Columns.User])
test.drop(test[test[rectools.Columns.User].isin(cold_users)].index, inplace=True)
hot_users = test[rectools.Columns.User].unique()
print(test.shape[0])
print(test[rectools.Columns.User].nunique())

def scorer(map: float):
    print(f"Your MAP: {map}")
    UPPER_BOUND = 0.095
    LOWER_BOUND = 0.075
    score = int(min(max( (map - LOWER_BOUND) / (UPPER_BOUND - LOWER_BOUND), 0), 1) * 60)
    print(f"Your Score: {score}")
    return score

In [ ]:
def solution(train: pd.DataFrame, users: pd.DataFrame, items: pd.DataFrame):
    #  CODE BEGIN
    # 1. Feature Engineering on Users and Items
    train_max_date = train[rectools.Columns.Datetime].max()
    train_train = train[(train[rectools.Columns.Datetime] <= train_max_date - pd.Timedelta(days=7))].copy()
    train_val = train[(train[rectools.Columns.Datetime] > train_max_date - pd.Timedelta(days=7))].copy()

    user_stats = train.groupby(rectools.Columns.User).agg(
        u_n_watches=('item_id', 'count'),
        u_mean_dur=(rectools.Columns.Weight, 'mean'),
        u_mean_watched_pct=('watched_pct', 'mean'),
    ).reset_index()

    item_stats = train.groupby(rectools.Columns.Item).agg(
        i_n_watches=('user_id', 'count'),
        i_mean_dur=(rectools.Columns.Weight, 'mean'),
        i_mean_watched_pct=('watched_pct', 'mean'),
    ).reset_index()

    item_pop_14 = train[train[rectools.Columns.Datetime] >= train_max_date - pd.Timedelta(days=14)][rectools.Columns.Item].value_counts().rename('i_pop_14').reset_index().rename(columns={'index': rectools.Columns.Item})
    item_pop_30 = train[train[rectools.Columns.Datetime] >= train_max_date - pd.Timedelta(days=30)][rectools.Columns.Item].value_counts().rename('i_pop_30').reset_index().rename(columns={'index': rectools.Columns.Item})
    item_pop_60 = train[train[rectools.Columns.Datetime] >= train_max_date - pd.Timedelta(days=60)][rectools.Columns.Item].value_counts().rename('i_pop_60').reset_index().rename(columns={'index': rectools.Columns.Item})

    user_meta = users[['user_id', 'age', 'income', 'sex', 'kids_flg']].fillna('Unknown')
    for col in ['age', 'income', 'sex', 'kids_flg']:
        user_meta[col] = user_meta[col].astype(str)

    item_meta = items[['item_id', 'content_type', 'release_year', 'for_kids', 'age_rating']].copy()
    item_meta['content_type'] = item_meta['content_type'].fillna('Unknown').astype(str)
    item_meta['for_kids'] = item_meta['for_kids'].fillna(-1).astype(str)
    item_meta['age_rating'] = item_meta['age_rating'].fillna(-1).astype(str)
    item_meta['release_year'] = item_meta['release_year'].fillna(-1)

    # 2. Fit Stage 1 Candidate Generators on Full Train
    recent_train = train[train[rectools.Columns.Datetime] >= train_max_date - pd.Timedelta(days=60)].copy()
    ds_pop = rectools.dataset.Dataset.construct(recent_train)
    pop = rectools.models.PopularModel()
    pop.fit(ds_pop)

    t_als = train.copy()
    t_als[rectools.Columns.Weight] = 1.0
    ds_als = rectools.dataset.Dataset.construct(t_als)
    als = rectools.models.ImplicitALSWrapperModel(
        model=implicit.als.AlternatingLeastSquares(factors=8, regularization=0.01, alpha=15.0, iterations=15, random_state=42, num_threads=4)
    )
    als.fit(ds_als)

    ease = rectools.models.EASEModel(regularization=500.0)
    ease.fit(ds_als)

    # Generate candidates for hot users
    recos_pop = pop.recommend(users=hot_users, dataset=ds_pop, k=40, filter_viewed=True)[[rectools.Columns.User, rectools.Columns.Item, 'rank', 'score']].rename(columns={'rank': 'rank_pop', 'score': 'score_pop'})
    recos_als = als.recommend(users=hot_users, dataset=ds_als, k=40, filter_viewed=True)[[rectools.Columns.User, rectools.Columns.Item, 'rank', 'score']].rename(columns={'rank': 'rank_als', 'score': 'score_als'})
    recos_ease = ease.recommend(users=hot_users, dataset=ds_als, k=40, filter_viewed=True)[[rectools.Columns.User, rectools.Columns.Item, 'rank', 'score']].rename(columns={'rank': 'rank_ease', 'score': 'score_ease'})

    test_cands = pd.concat([
        recos_pop[[rectools.Columns.User, rectools.Columns.Item]],
        recos_als[[rectools.Columns.User, rectools.Columns.Item]],
        recos_ease[[rectools.Columns.User, rectools.Columns.Item]],
    ]).drop_duplicates()

    test_cands = test_cands.merge(recos_pop, on=[rectools.Columns.User, rectools.Columns.Item], how='left')
    test_cands = test_cands.merge(recos_als, on=[rectools.Columns.User, rectools.Columns.Item], how='left')
    test_cands = test_cands.merge(recos_ease, on=[rectools.Columns.User, rectools.Columns.Item], how='left')

    # 3. Fit Candidate Generators on Split for CatBoost Training
    train_hot_users = train_val[train_val[rectools.Columns.User].isin(train_train[rectools.Columns.User])][rectools.Columns.User].unique()

    ds_train_pop = rectools.dataset.Dataset.construct(train_train[train_train[rectools.Columns.Datetime] >= train_train[rectools.Columns.Datetime].max() - pd.Timedelta(days=60)].copy())
    pop_train_model = rectools.models.PopularModel()
    pop_train_model.fit(ds_train_pop)

    t_train_als = train_train.copy()
    t_train_als[rectools.Columns.Weight] = 1.0
    ds_train_als = rectools.dataset.Dataset.construct(t_train_als)
    als_train_model = rectools.models.ImplicitALSWrapperModel(
        model=implicit.als.AlternatingLeastSquares(factors=8, regularization=0.01, alpha=15.0, iterations=15, random_state=42, num_threads=4)
    )
    als_train_model.fit(ds_train_als)

    ease_train_model = rectools.models.EASEModel(regularization=500.0)
    ease_train_model.fit(ds_train_als)

    tr_pop = pop_train_model.recommend(users=train_hot_users, dataset=ds_train_pop, k=40, filter_viewed=True)[[rectools.Columns.User, rectools.Columns.Item, 'rank', 'score']].rename(columns={'rank': 'rank_pop', 'score': 'score_pop'})
    tr_als = als_train_model.recommend(users=train_hot_users, dataset=ds_train_als, k=40, filter_viewed=True)[[rectools.Columns.User, rectools.Columns.Item, 'rank', 'score']].rename(columns={'rank': 'rank_als', 'score': 'score_als'})
    tr_ease = ease_train_model.recommend(users=train_hot_users, dataset=ds_train_als, k=40, filter_viewed=True)[[rectools.Columns.User, rectools.Columns.Item, 'rank', 'score']].rename(columns={'rank': 'rank_ease', 'score': 'score_ease'})

    train_cands = pd.concat([
        tr_pop[[rectools.Columns.User, rectools.Columns.Item]],
        tr_als[[rectools.Columns.User, rectools.Columns.Item]],
        tr_ease[[rectools.Columns.User, rectools.Columns.Item]],
    ]).drop_duplicates()

    train_cands = train_cands.merge(tr_pop, on=[rectools.Columns.User, rectools.Columns.Item], how='left')
    train_cands = train_cands.merge(tr_als, on=[rectools.Columns.User, rectools.Columns.Item], how='left')
    train_cands = train_cands.merge(tr_ease, on=[rectools.Columns.User, rectools.Columns.Item], how='left')

    val_pairs = set(zip(train_val[rectools.Columns.User], train_val[rectools.Columns.Item]))
    train_cands['target'] = [1 if (u, i) in val_pairs else 0 for u, i in zip(train_cands[rectools.Columns.User], train_cands[rectools.Columns.Item])]

    def add_features(df):
        df = df.merge(user_meta, on='user_id', how='left')
        df = df.merge(item_meta, on='item_id', how='left')
        df = df.merge(user_stats, on='user_id', how='left')
        df = df.merge(item_stats, on='item_id', how='left')
        df = df.merge(item_pop_14, on='item_id', how='left')
        df = df.merge(item_pop_30, on='item_id', how='left')
        df = df.merge(item_pop_60, on='item_id', how='left')
        
        df['rank_pop'] = df['rank_pop'].fillna(999)
        df['rank_als'] = df['rank_als'].fillna(999)
        df['rank_ease'] = df['rank_ease'].fillna(999)
        df['score_pop'] = df['score_pop'].fillna(0)
        df['score_als'] = df['score_als'].fillna(0)
        df['score_ease'] = df['score_ease'].fillna(0)
        df['i_pop_14'] = df['i_pop_14'].fillna(0)
        df['i_pop_30'] = df['i_pop_30'].fillna(0)
        df['i_pop_60'] = df['i_pop_60'].fillna(0)
        
        df['rrf_score'] = (
            1.0 / (30.0 + df['rank_pop']) +
            1.0 / (30.0 + df['rank_als']) +
            1.0 / (30.0 + df['rank_ease'])
        )
        return df

    train_df = add_features(train_cands)
    test_df = add_features(test_cands)

    feature_cols = [
        'rank_pop', 'rank_als', 'rank_ease',
        'score_pop', 'score_als', 'score_ease',
        'rrf_score',
        'age', 'income', 'sex', 'kids_flg',
        'content_type', 'for_kids', 'age_rating', 'release_year',
        'u_n_watches', 'u_mean_dur', 'u_mean_watched_pct',
        'i_n_watches', 'i_mean_dur', 'i_mean_watched_pct',
        'i_pop_14', 'i_pop_30', 'i_pop_60'
    ]
    cat_cols = ['age', 'income', 'sex', 'kids_flg', 'content_type', 'for_kids', 'age_rating']

    # 4. Train Stage 2 CatBoost Reranker
    cb = catboost.CatBoostClassifier(
        iterations=200,
        learning_rate=0.08,
        depth=6,
        cat_features=cat_cols,
        random_seed=42,
        verbose=0,
    )
    cb.fit(train_df[feature_cols], train_df['target'])

    test_df['cb_score'] = cb.predict_proba(test_df[feature_cols])[:, 1]

    # 5. Rerank and extract top 10
    test_df_sorted = test_df.sort_values(by=[rectools.Columns.User, 'cb_score'], ascending=[True, False])
    recos_final = test_df_sorted.groupby(rectools.Columns.User).head(10).copy()
    recos_final['rank'] = recos_final.groupby(rectools.Columns.User).cumcount() + 1
    #  CODE END
    return recos_final[[rectools.Columns.User, rectools.Columns.Item, 'rank']]


In [ ]:
%%time

recos = solution(train.copy(), users.copy(), items.copy())
scorer(rectools.metrics.MAP(10).calc(recos, test))